In [86]:
import gc
gc.collect()

2213

In [87]:
import nibabel as nib
from pathlib import Path
import pickle
import os

In [88]:
data_folder =  '/home/robakp/Exeriments1/prostate_lesion_detection/rjozwiak-MGR_dataset_correct/MGR_dataset_correct'

channels = {
    'adc' : 'adc',
    'anatomy' : 'anatomy',
    'dwi' : 'dwi',
    't2' : 't2'
}

target = 'lesion'

file_extention = '.nii.gz'

preprocessed_steps = [
    'raw',
    'filling_anatomy_gaps'
]


In [89]:
def get_all_patients_ids(data_path):
    folder = Path(data_folder)
    ids = []
    for patient in folder.iterdir():
        ids.append(patient.name)
    return ids


In [90]:

class preprocess_file_manager:
    def __init__(self, main_folder,preprocess_steps):
        self.main_folder = main_folder
        self.steps = preprocess_steps

    def load_file(self, step, patient_id):
        path = os.path.join(self.main_folder, step, f"{patient_id}.pkl")

        with open(path, "rb") as f:
            data = pickle.load(f)

        return data

    def save_file(self, step, patient_id, data):
        path = os.path.join(self.main_folder, step, f"{patient_id}.pkl")
        
        os.makedirs(os.path.dirname(path), exist_ok=True)

        with open(path, "wb") as f:
            pickle.dump(data, f)

In [91]:
def load_patient(data_path,patient):
    patient_data = {}
    patient_path = data_path / Path(patient)
    for key in channels:
        channel_file = patient_path / f"{channels[key]}.nii.gz"
        df = nib.load(channel_file)
        ##
        patient_data[key] = df.get_fdata()
    return patient_data

In [92]:
def load_raw_data(data_folder):
    data = {}
    folder = Path(data_folder)

    for patient in folder.iterdir():
        ##
        if patient.is_dir():
            patient_data = load_patient(data_folder,patient.name)
            ##
            data[patient.name] = patient_data
    return data


In [94]:
patients = get_all_patients_ids(data_folder)

dispaly

In [96]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [97]:
def normalize_volume(volume):
    vmin = volume.min()
    vmax = volume.max()
    if vmax - vmin == 0:
        return np.zeros_like(volume)
    return (volume - vmin) / (vmax - vmin)

In [149]:
channels = {
    'adc': 'adc',
    'anatomy': 'anatomy',
    'dwi': 'dwi',
    't2': 't2'
}

def show_image(patient, channel, slice_idx):
    plt.figure(figsize=(5,5))
    plt.imshow(
        normalize_volume(load_patient(data_folder,patient)[channel][:, :, slice_idx]),
        cmap='gray',
        vmin=0,
        vmax=1
    )
    plt.title(f"Patient: {patient} | Channel: {channel} | Slice: {slice_idx}")
    plt.axis('off')
    plt.show()

# Patient Dropdown (NEW)
patient_dropdown = widgets.Dropdown(
    options=sorted(patients),
    value=sorted(patients)[0],
    description='Patient:'
)

# Slice Slider
slice_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=50,  # You may later want to update this dynamically per patient
    step=1,
    description='Slice:',
    continuous_update=False
)

# Channel Dropdown
channel_dropdown = widgets.Dropdown(
    options=list(channels.keys()),
    value='adc',
    description='Channel:'
)

# Interactive display
widgets.interactive(
    show_image,
    patient=patient_dropdown,
    channel=channel_dropdown,
    slice_idx=slice_slider
)

interactive(children=(Dropdown(description='Patient:', options=('001', '003', '004', '005', '007', '008', '011…

<Figure size 500x500 with 0 Axes>

remember about allingning

How to detect if prostate even is on picture? Is this in data? I think it is

In [99]:
file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps)

In [100]:

folder = Path(data_folder)

for patient in folder.iterdir():
    if patient.is_dir():
        patient_data = load_patient(data_folder,patient.name)
        file_manager.save_file('raw', patient.name, patient_data)

check how big portions with prostate are

In [108]:
def find_periods(arr):
    if len(arr) == 0:
        return []

    periods = []
    start = arr[0]
    prev = arr[0]

    for num in arr[1:]:
        if num == prev + 1:
            prev = num
        else:
            periods.append((start, prev))
            start = num
            prev = num

    periods.append((start, prev))
    return periods

In [102]:
for patient in patients[:1]:
    print(patient)
    prostate = file_manager.load_file('raw',patient)['anatomy'] 
    valid_layers = np.any(prostate == 1, axis=(0, 1))
    print(valid_layers.shape)

3322
(41,)


Working with holes in prostate layer

In [103]:
true_indices = np.where(valid_layers)[0]
print(true_indices)
print(len(find_periods(true_indices)))

[15 16 17 18 19 20 22 23 24 25 26 27 28 29 30]
2


In [115]:
def find_gaps_in_anatomy(patients,file_manager,step = 'raw'):
    outliers = []
    sections_with_prostate = {}
    for patient in patients:
        prostate = file_manager.load_file(step, patient)['anatomy'] 
        valid_layers = np.any(prostate == 1, axis=(0, 1))
        true_indices = np.where(valid_layers)[0]
        if len(find_periods(true_indices)) != 1:
            outliers.append(patient)
        periods = find_periods(true_indices)
        if len(periods) >= 0:
            sections_with_prostate[patient] = (periods[0][0], periods[-1][1])
        else:
            raise ValueError(f"Patient {patient} without anatomy data")
    return outliers,sections_with_prostate

outliers,sections_with_prostate = find_gaps_in_anatomy(patients,file_manager)

In [118]:
for outlier in outliers:
    print(outlier)
    prostate = file_manager.load_file('raw',outlier)['anatomy'] 
    valid_layers = np.any(prostate == 1, axis=(0, 1))
    true_indices = np.where(valid_layers)[0]
    periods = find_periods(true_indices)
    print(len(periods))
    print(periods)

3322
2
[(15, 20), (22, 30)]
432
2
[(14, 17), (19, 25)]
524
2
[(13, 18), (21, 22)]
798
2
[(13, 19), (21, 21)]
696
2
[(13, 17), (19, 23)]
1038
2
[(16, 18), (20, 26)]
534
2
[(17, 17), (19, 32)]
1183
2
[(9, 13), (15, 18)]
906
2
[(9, 14), (16, 18)]
655
2
[(15, 19), (21, 29)]


In [120]:
import numpy as np
from scipy.ndimage import distance_transform_edt

def signed_distance(mask):
    mask = mask.astype(bool)
    outside = distance_transform_edt(~mask)
    inside = distance_transform_edt(mask)
    return outside - inside

def interpolate_shapes(A, B, steps):
    dA = signed_distance(A)
    dB = signed_distance(B)
    result = []
    for i in range(1, steps + 1):
        t = i / (steps + 1)
        d = (1 - t) * dA + t * dB
        result.append((d < 0).astype(int))
    return result

A = np.array([[0, 1, 1, 1, 1, 1],
              [0, 1, 1, 1, 1, 1],
              [0, 1, 1, 1, 1, 1],
              [0, 1, 1, 1, 1, 1],
              [0, 1, 1, 1, 1, 1],
              [0, 0, 0, 0, 0, 0]])

B = np.array([[0, 0, 0, 0, 0, 1],
              [0, 0, 0, 0, 0, 0],
              [0, 0, 0, 0, 0, 0],
              [0, 0, 0, 0, 0, 0],
              [0, 0, 0, 0, 0, 0],
              [0, 0, 0, 0, 0, 0]])

frames = interpolate_shapes(A, B, 3)

In [129]:
outlier = '524'
def fix_patient_anatomy(outlier, file_manager,step = 'raw'):
    outlier_data = file_manager.load_file(step, outlier)
    prostate = outlier_data['anatomy']
    valid_layers = np.any(prostate == 1, axis=(0, 1))
    true_indices = np.where(valid_layers)[0]
    i = 0
    periods = find_periods(true_indices)
    while i < len(periods)-1:
        start = periods[i][1]
        end = periods[i+1][0]
        print(f"{start}  {end}")

        start_layer = prostate[:,:,start]
        end_layer = prostate[:,:,end]
        number_of_interpolation_layers = end-start-1

        new_leayers = interpolate_shapes(start_layer,end_layer,number_of_interpolation_layers)

        j = start+1
        while j <  end:
            prostate[:,:,j] = new_leayers[j-start-1]
            j+=1
        i+=1
    file_manager.save_file(step,outlier,outlier_data)

    

fix_patient_anatomy(outlier, file_manager)

18  21


In [133]:
outliers, _ = find_gaps_in_anatomy(patients,file_manager)
print(outliers)
for outlier in outliers:
    fix_patient_anatomy(outlier,file_manager)

e, sections_with_prostate  = find_gaps_in_anatomy(patients,file_manager)
print(e)

[]
[]


Cutting pictures into correct sizes

In [139]:
differences = [b-a+1 for a, b in sections_with_prostate.values()]
print(differences)

[16, 8, 12, 15, 10, 10, 11, 14, 14, 16, 10, 10, 10, 5, 19, 13, 14, 17, 11, 17, 11, 10, 16, 11, 12, 12, 12, 14, 14, 14, 13, 11, 14, 13, 11, 12, 17, 12, 12, 12, 11, 15, 12, 16, 13, 9, 10, 11, 14, 8, 16, 11, 17, 12, 12, 13, 17, 14, 22, 15, 19, 15, 12, 17, 13, 12, 12, 12, 11, 15, 14, 10, 12, 10, 12, 13, 15, 13, 10, 20, 13, 16, 14, 14, 11, 8, 11, 13, 12, 14, 12, 16, 17, 12, 11, 12, 13, 11, 12, 13, 17, 14, 14, 12, 16, 6, 8, 11, 9, 10, 10, 19, 12, 16, 11, 14, 13, 18, 12, 13, 16, 10, 10, 14, 9, 12, 9, 14, 15, 16, 14, 11, 16, 12, 12, 10, 11, 16, 16, 10, 10, 9, 11, 12, 8, 19, 14, 9, 17, 10, 12, 12, 11, 11, 16, 21, 14, 13, 13, 14, 19, 13, 13, 12, 10, 17, 14, 10, 14, 15, 20, 10, 20, 23, 14, 12, 9, 22, 9, 12, 12, 27, 17, 12, 11, 16, 14, 18, 14, 15, 25, 8, 12, 13, 9, 9, 18, 13, 12, 10, 10, 14, 15, 12, 21, 8, 9, 13, 15, 10, 9, 12, 12, 13, 14, 11, 15, 13, 9, 11, 14, 14, 11, 9, 10, 15, 13, 15, 10, 12, 15, 12, 17, 9, 10, 10, 10, 11, 13, 16, 17, 9, 19, 21, 11, 14, 10, 14, 16, 13, 11, 12, 13, 9, 11, 17, 1

In [148]:
biggest_gap = (0, -1, 0)
smallest_gap = (0,99999, 0)
for patient in sections_with_prostate:
    diff = sections_with_prostate[patient][1] - sections_with_prostate[patient][0] + 1
    if diff > biggest_gap[1]:
        biggest_gap = (patient,diff,sections_with_prostate[patient])
    if diff < smallest_gap[1]:
        smallest_gap = (patient,diff,sections_with_prostate[patient])

print(biggest_gap)
print(smallest_gap)

('482', 27, (13, 39))
('3071', 4, (15, 18))


looking for smallest picture

In [159]:
smallest_number_of_slices = 9999
for patient in patients:
    data = file_manager.load_file('raw', patient)
    shape_per_channel = [data[channel].shape for channel in channels]
    all_same = len(set(shape_per_channel)) == 1
    if not all_same:
        print(f"ALERT ALERT {patient}")
    number_of_slices = shape_per_channel[0][2]
    # print(number_of_slices)
    if smallest_number_of_slices > number_of_slices:
        smallest_number_of_slices = number_of_slices

print(f"smallest number of slices {smallest_number_of_slices}")
    

smallest number of slices 25
